# Calibrating and Summarizing Sensor Measurements

*Combine a linear solve, matrix-vector multiplication, gradients, statistics, and seeded sampling in one measurement report.*


## Problem Definition

### Question

Two temperature sensors record readings every minute. Apply a shared
calibration rule, combine their estimates, and report how the combined
temperature changes over time.

| Time (minutes) | Sensor A (raw units) | Sensor B (raw units) |
|---:|---:|---:|
| 0 | 10 | 12 |
| 1 | 12 | 14 |
| 2 | 15 | 16 |
| 3 | 18 | 18 |
| 4 | 20 | 20 |

The calibration checks pair a raw reading of 10 with 21 degrees
Celsius and a raw reading of 20 with 41 degrees Celsius. Assume
`temperature = gain * raw_reading + offset` for both sensors.
Combine their calibrated readings using the supplied weights 0.6
for sensor A and 0.4 for sensor B.

### Constraints

- Solve for the gain and offset from the two calibration checks.
- Retain rows as times and columns as sensors.
- Use `@` for the weighted estimate at every time.
- Calculate rates in degrees Celsius per minute.
- Describe these five readings with `ddof=0` standard deviation.
- Select two distinct time positions using seed 42 for inspection.


## Session Method Map

| Guided topic | Role in this report |
|---|---|
| [Vector products](01_vector_products.ipynb) | A dot product combines two sensor values |
| [Matrix operations](02_matrix_operations.ipynb) | Apply the weights to every row with `@` |
| [Linear equations](03_solving_linear_equations.ipynb) | Solve for gain and offset |
| [Gradients](04_gradients.ipynb) | Estimate temperature change per minute |
| [Descriptive statistics](05_descriptive_statistics.ipynb) | Summarize the combined series |
| [Correlation and covariance](06_correlation_and_covariance.ipynb) | Compare the two calibrated sensor columns |
| [Random sampling](07_random_sampling.ipynb) | Select two observations for inspection |


## Program Flow

```text
Prepared readings -> calibration solve -> calibrated table -> weighted series
                  -> rates and statistics -> seeded inspection sample -> report
```


## Imports and Prepared Measurements

The coefficient columns represent gain and offset, so each calibration
row contains `[raw_reading, 1]`. The weight vector follows the same
sensor order as the reading columns.

**Question.** Will the weighted value at minute 0 be 23.0 degrees
Celsius, the simple average of the two calibrated readings?


In [1]:
import numpy as np

times = np.arange(5, dtype=float)
raw_readings = np.array([[10, 12],
                         [12, 14],
                         [15, 16],
                         [18, 18],
                         [20, 20]], dtype=float)
calibration_coefficients = np.array([[10, 1],
                                     [20, 1]], dtype=float)
reference_temperatures = np.array([21, 41], dtype=float)
weights = np.array([0.6, 0.4])

print("Reading shape:", raw_readings.shape)
print("Weights:", weights)
print("Weight sum:", weights.sum())


Reading shape: (5, 2)
Weights: [0.6 0.4]
Weight sum: 1.0


There are five times and two sensors. The two weights sum to 1, but
they give more influence to sensor A than to sensor B.


## Solving and Applying the Calibration

Solve the two equations for gain and offset, then apply those scalar
values to the entire reading table. Return the calibrated data along
with the parameters so the report can verify the calibration checks.


In [2]:
def calibrate_readings(readings, coefficients, references):
    gain, offset = np.linalg.solve(coefficients, references)
    calibrated = gain * readings + offset
    return calibrated, gain, offset


## Combining and Summarizing Measurements

`(5, 2) @ (2,)` gives a five-value weighted series. The time coordinates
supply the spacing to `np.gradient()`. Each sensor occupies a column,
so its relationship matrices use `rowvar=False`.


In [3]:
def summarize_measurements(calibrated, sample_times, sensor_weights):
    combined = calibrated @ sensor_weights
    return {
        "combined": combined,
        "rates": np.gradient(combined, sample_times),
        "mean": combined.mean(),
        "median": np.median(combined),
        "std": combined.std(ddof=0),
        "correlation": np.corrcoef(calibrated, rowvar=False),
        "covariance": np.cov(calibrated, rowvar=False, ddof=1),
    }


## Selecting Observations for Inspection

Use a generator inside the helper so each complete workflow call
reproduces the same two positions. The sample is a selection for
inspection; the report's statistics still use all five observations.


In [4]:
def select_inspection_indices(count):
    rng = np.random.default_rng(42)
    return rng.choice(np.arange(count), size=2, replace=False)


## Formatting the Report

Report temperature values and rates with their units. Read the
off-diagonal entry `[0, 1]` to compare the two sensor columns.


In [5]:
def print_report(calibrated, summary, selected_indices):
    print("Calibrated readings (degrees Celsius):")
    print(calibrated)
    print("Combined temperatures:", np.round(summary["combined"], 2))
    print("Rates (degrees Celsius per minute):", np.round(summary["rates"], 2))
    print(f"Mean temperature: {summary['mean']:.2f}")
    print(f"Median temperature: {summary['median']:.2f}")
    print(f"Standard deviation (ddof=0): {summary['std']:.2f}")
    print(f"Sensor correlation: {summary['correlation'][0, 1]:.4f}")
    print(f"Sensor covariance (ddof=1): {summary['covariance'][0, 1]:.2f}")
    print("Inspection times (minutes):", times[selected_indices])
    print("Inspection temperatures:", np.round(summary["combined"][selected_indices], 2))


## Running the Workflow

Verify the two reference temperatures by substitution before
reporting. The returned values allow the computed results to be
inspected after `main()` completes.


In [6]:
def main():
    calibrated, gain, offset = calibrate_readings(
        raw_readings, calibration_coefficients, reference_temperatures
    )
    parameters = np.array([gain, offset])
    print(f"Calibration gain: {gain:.2f}; offset: {offset:.2f}")
    print(
        "Calibration checks match:",
        np.allclose(calibration_coefficients @ parameters, reference_temperatures),
    )

    summary = summarize_measurements(calibrated, times, weights)
    selected_indices = select_inspection_indices(times.size)
    print_report(calibrated, summary, selected_indices)
    return calibrated, summary, selected_indices


calibrated_readings, report, inspection_indices = main()


Calibration gain: 2.00; offset: 1.00
Calibration checks match: True
Calibrated readings (degrees Celsius):
[[21. 25.]
 [25. 29.]
 [31. 33.]
 [37. 37.]
 [41. 41.]]
Combined temperatures: [22.6 26.6 31.8 37.  41. ]
Rates (degrees Celsius per minute): [4.  4.6 5.2 4.6 4. ]
Mean temperature: 31.80
Median temperature: 31.80
Standard deviation (ddof=0): 6.68
Sensor correlation: 0.9971
Sensor covariance (ddof=1): 52.00
Inspection times (minutes): [0. 3.]
Inspection temperatures: [22.6 37. ]


## Result Analysis

The gain is **2.00** and the offset is **1.00**; both calibration
checks match. The calibrated readings at minute 0 are 21 and 25
degrees Celsius. The prediction is false: their weighted result is
`21 * 0.6 + 25 * 0.4 = 22.6`, rather than the unweighted average 23.0.

The combined series is `[22.6, 26.6, 31.8, 37.0, 41.0]`. Its mean and
median are both 31.80 degrees Celsius, and its `ddof=0` standard
deviation is about 6.68 degrees Celsius. Estimated rates are
`[4.0, 4.6, 5.2, 4.6, 4.0]` degrees Celsius per minute; the largest
estimate occurs at minute 2.

The sensor correlation is about 0.9971 and their sample covariance
is 52.00 degrees Celsius squared. They change together closely, but
high correlation does not mean that their readings agree at every time.

The seeded selection chooses minutes 0 and 3, with combined
temperatures 22.6 and 37.0 degrees Celsius. The summary continues to
describe all five observations, independently of that selection.
